# DataFrame 사용자 정의 함수


---
## 1. 환경 설정

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
print(f"Pandas 버전: {pd.__version__}")

In [ ]:
# Titanic 데이터셋 로드
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)
print(f"데이터 크기: {titanic.shape}")
titanic.head()

---
## 2. 사용자 함수 (User Defined Function) 개요

### 개념
- 내장 함수 외 사용자가 직접 설계한 **맞춤형 연산 모듈**
- Pandas DataFrame의 특성 데이터에 맞는 **커스텀 로직**이 필수적

### 목적
- DataFrame 내 반복 연산 **최소화**
- 커스텀 데이터 분석의 **모듈화** 및 **재사용성** 확보

---
## 3. 커스텀 엔진 구조: 함수 정의 (Definition)

```python
def 함수명(매개변수1, 매개변수2, ...):
    """함수 설명"""
    ...  # 실행 블록 구분 (들여쓰기/indent)
    return 반환값
```

- `def`: 함수 선언 키워드
- `return`: DataFrame 매핑 시 필수! Call(호출)에 대한 **결과를 반환하는 구문**

---
## 4. 커스텀 엔진 구동: 함수 호출 (Call)

```python
함수명(전달 인자)
```

- 정의된 메모리 주소의 **연산 로직 실행**
- DataFrame 연산 시, `.apply(함수명)` 형태로 **간접 호출** 활용
- 괄호 `()` 내에 실제 데이터(전달 인자) 주입

---
## 5. 함수 설계 핵심 용어

### 매개변수 (Parameter) vs 전달 인자 (Argument)

| 구분 | 설명 | DataFrame 연계 |
|------|------|---------------|
| **매개변수 (Parameter)** | 함수에 입력으로 전달된 값을 받는 변수 (빈 공간) | DataFrame 열(Column) — 데이터가 매핑될 타겟 |
| **전달 인자 (Argument)** | 함수를 호출할 때 전달하는 **실제 값** | DataFrame 연산 시 각 행(Row)의 **실제 데이터** |

### 기본 매개변수 (Default Parameter)
- 전달 인자값이 없을 경우 **기본값을 받는 변수**
- 결측치(NaN) 처리 및 DataFrame 연산 시 유연성 제공
- 구조: `매개변수=기본값` 형태 선언

---
## 6. DataFrame 연산용 함수 4대 유형 분석

| 유형 | 입력(매개변수) | 반환(return) | 특징 |
|------|-------------|-------------|------|
| **Type 1** | 입력(O) / 반환(O) | O | 표준 매핑 구조(표준형) |
| **Type 2** | 입력(X) / 반환(O) | O | 정적 데이터 타입 생성 |
| **Type 3** | 입력(O) / 반환(X) | X | 결과 출력 전용 (값 축적 주의) |
| **Type 4** | 입력(X) / 반환(X) | X | 단순 실행 트리거 (매핑 불가) |

---
## 7. Type 1: 입력값(O) & 반환값(O) 모델

### 특징
- 외부 데이터 수용 및 최종 결과를 **외부 송출**
- DataFrame 연산 로직의 **표준 설계 형태**
- `return`이 있어야 `.apply()` 적용 시 새로운 Column으로 정상 매핑

In [ ]:
# Type 1: 입력(O), 반환(O) - 표준형
def calc_family_size(sibsp, parch):
    """SibSp와 Parch를 합산하여 가족 수 반환"""
    res = sibsp + parch + 1
    return res  # return: sibsp + parch + 1 (본인 포함)

# 호출 및 결과
res = calc_family_size(1, 2)
print(f"calc_family_size(1, 2) = {res}")  # 출력: 4

### Type 1: DataFrame 매핑 연계

- `return` 값이 존재하여 **변수 할당 연계 지원**
- DataFrame `.apply()` 적용 시 연산 결과가 새로운 Column으로 **정상 매핑**

In [ ]:
# Type 1: DataFrame 매핑 실습
df = titanic[['Name', 'SibSp', 'Parch']].head(5).copy()
print("원본 DataFrame:")
print(df)

# calc_family_size 함수를 DataFrame에 적용 (SibSp와 Parch의 합 + 1)
df['FamilySize'] = df.apply(lambda row: calc_family_size(row['SibSp'], row['Parch']), axis=1)
print("\ncalc_family_size(SibSp, Parch) 결과 → FamilySize:")
print(df)

In [ ]:
# Type 1: 다양한 활용 예시
def calc_adjusted_fare(fare, pclass, discount_rate=0.1):
    """운임에 등급별 할인율을 적용하여 반환"""
    return fare * (1 - discount_rate * (pclass - 1))

df2 = titanic[['Name', 'Pclass', 'Fare']].head(5).copy()
df2['AdjustedFare'] = df2.apply(lambda row: calc_adjusted_fare(row['Fare'], row['Pclass']), axis=1)
print("calc_adjusted_fare(Fare, Pclass, discount_rate=0.1):")
print(df2)